In [19]:
"""
DGCI&S EXIM Data Preprocessing Pipeline
Processes raw export-import data for critical minerals analysis
Handles: Copper (HSN 2603), Lithium (HSN 2530), Graphite (HSN 2508)
"""

'\nDGCI&S EXIM Data Preprocessing Pipeline\nProcesses raw export-import data for critical minerals analysis\nHandles: Copper (HSN 2603), Lithium (HSN 2530), Graphite (HSN 2508)\n'

In [20]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

============================================================================
SECTION 1: DATA LOADING AND CLEANING
============================================================================

In [21]:
class DGCISDataLoader:
    """Load and parse DGCI&S data files"""
    
    def __init__(self):
        self.raw_data = None
        self.hsn_mapping = {
            '2603': 'Copper',
            '260300': 'Copper ores and concentrates',
            '2530': 'Lithium',
            '253090': 'Mineral substances containing lithium',
            '2508': 'Graphite',
            '250810': 'Natural graphite in powder or flakes',
            '250890': 'Other graphite'
        }
        
    def load_excel(self, file_path, sheet_name=0):
        """Load DGCI&S Excel file"""
        print(f"Loading data from: {file_path}")
        
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name)
            print(f"✓ Loaded {len(df)} rows, {len(df.columns)} columns")
            self.raw_data = df
            return df
        except Exception as e:
            print(f"✗ Error loading file: {e}")
            return None
    
    def load_csv(self, file_path, encoding='utf-8'):
        """Load DGCI&S CSV file"""
        print(f"Loading data from: {file_path}")
        
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✓ Loaded {len(df)} rows, {len(df.columns)} columns")
            self.raw_data = df
            return df
        except Exception as e:
            print(f"✗ Error loading file: {e}")
            # Try alternate encoding
            try:
                df = pd.read_csv(file_path, encoding='latin-1')
                print(f"✓ Loaded with latin-1 encoding")
                self.raw_data = df
                return df
            except:
                return None
    
    def inspect_data(self, df=None):
        """Inspect data structure and quality"""
        if df is None:
            df = self.raw_data
            
        print("\n" + "="*70)
        print("DATA INSPECTION REPORT")
        print("="*70)
        
        print(f"\nShape: {df.shape}")
        print(f"\nColumn Names:")
        for idx, col in enumerate(df.columns, 1):
            print(f"{idx}. {col}")
        
        print(f"\nData Types:")
        print(df.dtypes)
        
        print(f"\nMissing Values:")
        missing = df.isnull().sum()
        print(missing[missing > 0])
        
        print(f"\nFirst 5 rows:")
        print(df.head())
        
        return df.info()

In [22]:
class DGCISDataCleaner:
    """Clean and standardize DGCI&S data"""
    
    @staticmethod
    def standardize_columns(df):
        """Standardize column names"""
        # Common DGCI&S column name variations
        column_mapping = {
            'HSCode': 'hsn_code',
            'HS Code': 'hsn_code',
            'HSN': 'hsn_code',
            'Commodity': 'commodity_desc',
            'Description': 'commodity_desc',
            'Country': 'country',
            'Partner Country': 'country',
            'Year': 'year',
            'Month': 'month',
            'Value (Rs Cr)': 'value_inr_cr',
            'Value (USD)': 'value_usd',
            'Quantity': 'quantity',
            'Unit': 'unit',
            'Import/Export': 'trade_type',
            'Type': 'trade_type'
        }
        
        # Create new column names
        new_columns = []
        for col in df.columns:
            standardized = column_mapping.get(col.strip(), col.strip().lower().replace(' ', '_'))
            new_columns.append(standardized)
        
        df.columns = new_columns
        print("✓ Columns standardized")
        
        return df
    
    @staticmethod
    def clean_hsn_codes(df, hsn_column='hsn_code'):
        """Clean and standardize HSN codes"""
        if hsn_column not in df.columns:
            print(f"✗ Column '{hsn_column}' not found")
            return df
        
        # Remove spaces, convert to string
        df[hsn_column] = df[hsn_column].astype(str).str.replace(' ', '').str.strip()
        
        # Extract first 4 digits for chapter classification
        df['hsn_chapter'] = df[hsn_column].str[:4]
        
        # Filter for critical minerals (26, 25 chapters)
        df['is_critical_mineral'] = df['hsn_chapter'].isin(['2603', '2530', '2508'])
        
        print(f"✓ HSN codes cleaned. Found {df['is_critical_mineral'].sum()} critical mineral records")
        
        return df
    
    @staticmethod
    def parse_dates(df, year_col='year', month_col='month'):
        """Parse and create date columns"""
        if year_col in df.columns and month_col in df.columns:
            # Handle financial year format (2017-18)
            if df[year_col].dtype == 'object' and '-' in str(df[year_col].iloc[0]):
                df['financial_year'] = df[year_col]
                df[year_col] = df[year_col].str.split('-').str[0].astype(int)
            
            # Create date column
            df['date'] = pd.to_datetime(
                df[year_col].astype(str) + '-' + df[month_col].astype(str).str.zfill(2) + '-01',
                errors='coerce'
            )
            
            # Add quarter
            df['quarter'] = df['date'].dt.quarter
            
            print(f"✓ Date columns created")
        else:
            print(f"✗ Required date columns not found")
        
        return df
    
    @staticmethod
    def clean_numeric_values(df, value_columns=['value_inr_cr', 'quantity']):
        """Clean and convert numeric columns"""
        for col in value_columns:
            if col in df.columns:
                # Remove commas, convert to numeric
                df[col] = pd.to_numeric(
                    df[col].astype(str).str.replace(',', '').str.replace('$', ''),
                    errors='coerce'
                )
                
                # Handle negative values (returns)
                df[col] = df[col].abs()
                
                print(f"✓ Cleaned numeric column: {col}")
        
        return df
    
    @staticmethod
    def handle_missing_values(df, strategy='interpolate'):
        """Handle missing values in dataset"""
        print(f"\nMissing values before cleaning:")
        print(df.isnull().sum())
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        if strategy == 'interpolate':
            for col in numeric_cols:
                df[col] = df[col].interpolate(method='linear')
        elif strategy == 'forward_fill':
            df[numeric_cols] = df[numeric_cols].fillna(method='ffill')
        elif strategy == 'drop':
            df = df.dropna()
        
        print(f"\n✓ Missing values handled using '{strategy}' strategy")
        print(f"Missing values after cleaning:")
        print(df.isnull().sum())
        
        return df
    
    @staticmethod
    def remove_outliers(df, column, method='iqr', threshold=3):
        """Remove outliers from numeric columns"""
        if column not in df.columns:
            return df
        
        initial_count = len(df)
        
        if method == 'iqr':
            Q1 = df[column].quantile(0.25)
            Q3 = df[column].quantile(0.75)
            IQR = Q3 - Q1
            
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
            
        elif method == 'zscore':
            z_scores = np.abs((df[column] - df[column].mean()) / df[column].std())
            df = df[z_scores < threshold]
        
        removed_count = initial_count - len(df)
        print(f"✓ Removed {removed_count} outliers from '{column}' using {method} method")
        
        return df

============================================================================
SECTION 2: DATA TRANSFORMATION
============================================================================

In [23]:
class DGCISDataTransformer:
    """Transform and aggregate DGCI&S data"""
    
    @staticmethod
    def filter_critical_minerals(df, minerals=['Copper', 'Lithium', 'Graphite']):
        """Filter data for specific critical minerals"""
        hsn_codes = {
            'Copper': ['2603', '260300'],
            'Lithium': ['2530', '253090'],
            'Graphite': ['2508', '250810', '250890']
        }
        
        # Flatten all HSN codes
        target_codes = []
        for mineral in minerals:
            target_codes.extend(hsn_codes.get(mineral, []))
        
        # Filter
        df_filtered = df[df['hsn_chapter'].isin(target_codes) | 
                        df['hsn_code'].isin(target_codes)].copy()
        
        # Add mineral name
        df_filtered['mineral'] = df_filtered['hsn_code'].apply(
            lambda x: next((k for k, v in hsn_codes.items() if x in v), 'Other')
        )
        
        print(f"✓ Filtered {len(df_filtered)} records for {len(minerals)} minerals")
        
        return df_filtered
    
    @staticmethod
    def aggregate_by_time(df, freq='M', group_cols=['mineral', 'trade_type']):
        """Aggregate data by time period"""
        agg_dict = {
            'value_inr_cr': 'sum',
            'quantity': 'sum'
        }
        
        if 'date' not in df.columns:
            print("✗ Date column not found")
            return df
        
        df = df.set_index('date')
        
        # Group and resample
        df_agg = df.groupby(group_cols).resample(freq).agg(agg_dict).reset_index()
        
        print(f"✓ Data aggregated by {freq} frequency")
        
        return df_agg
    
    @staticmethod
    def pivot_import_export(df):
        """Create separate columns for import and export"""
        if 'trade_type' not in df.columns:
            print("✗ 'trade_type' column not found")
            return df
        
        # Pivot
        df_pivot = df.pivot_table(
            index=['date', 'mineral'],
            columns='trade_type',
            values='value_inr_cr',
            aggfunc='sum'
        ).reset_index()
        
        # Rename columns
        df_pivot.columns.name = None
        if 'Import' in df_pivot.columns:
            df_pivot = df_pivot.rename(columns={'Import': 'import_value'})
        if 'Export' in df_pivot.columns:
            df_pivot = df_pivot.rename(columns={'Export': 'export_value'})
        
        # Fill missing values
        df_pivot = df_pivot.fillna(0)
        
        # Calculate trade balance
        if 'import_value' in df_pivot.columns and 'export_value' in df_pivot.columns:
            df_pivot['trade_balance'] = df_pivot['export_value'] - df_pivot['import_value']
            df_pivot['dependency_ratio'] = (
                df_pivot['import_value'] / (df_pivot['import_value'] + df_pivot['export_value']) * 100
            )
        
        print("✓ Import-Export data pivoted")
        
        return df_pivot
    
    @staticmethod
    def calculate_growth_rates(df, value_column='import_value'):
        """Calculate YoY and MoM growth rates"""
        if value_column not in df.columns:
            return df
        
        # Sort by date
        df = df.sort_values('date')
        
        # Calculate growth rates
        df['mom_growth'] = df.groupby('mineral')[value_column].pct_change() * 100
        df['yoy_growth'] = df.groupby('mineral')[value_column].pct_change(12) * 100
        
        # Moving averages
        df['ma_3m'] = df.groupby('mineral')[value_column].rolling(3).mean().reset_index(0, drop=True)
        df['ma_12m'] = df.groupby('mineral')[value_column].rolling(12).mean().reset_index(0, drop=True)
        
        print(f"✓ Growth rates and moving averages calculated")
        
        return df
    
    @staticmethod
    def create_features(df):
        """Create additional features for analysis"""
        if 'date' not in df.columns:
            return df
        
        # Time features
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear
        
        # Cyclical features
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        
        # Lag features
        for lag in [1, 3, 6, 12]:
            df[f'lag_{lag}m'] = df.groupby('mineral')['import_value'].shift(lag)
        
        print("✓ Additional features created")
        
        return df

============================================================================
SECTION 3: COUNTRY-LEVEL ANALYSIS
============================================================================

In [24]:
class CountryAnalyzer:
    """Analyze trading partner data"""
    
    @staticmethod
    def get_top_partners(df, mineral, n=10, trade_type='Import'):
        """Get top trading partners for a mineral"""
        if 'country' not in df.columns:
            print("✗ 'country' column not found")
            return None
        
        df_filtered = df[
            (df['mineral'] == mineral) & 
            (df['trade_type'] == trade_type)
        ]
        
        top_partners = df_filtered.groupby('country').agg({
            'value_inr_cr': 'sum',
            'quantity': 'sum'
        }).sort_values('value_inr_cr', ascending=False).head(n)
        
        # Calculate share
        total_value = top_partners['value_inr_cr'].sum()
        top_partners['share_percent'] = (top_partners['value_inr_cr'] / total_value * 100).round(2)
        
        return top_partners
    
    @staticmethod
    def calculate_concentration_index(df, mineral):
        """Calculate Herfindahl-Hirschman Index for supply concentration"""
        partners = CountryAnalyzer.get_top_partners(df, mineral, n=100)
        
        if partners is None or len(partners) == 0:
            return None
        
        # Calculate HHI
        hhi = (partners['share_percent'] ** 2).sum()
        
        interpretation = "Low" if hhi < 1500 else "Moderate" if hhi < 2500 else "High"
        
        return {
            'HHI': round(hhi, 2),
            'Concentration': interpretation,
            'Top_5_Share': partners.head(5)['share_percent'].sum()
        }

============================================================================
SECTION 4: DATA EXPORT
============================================================================

In [25]:
class DataExporter:
    """Export processed data in various formats"""
    
    @staticmethod
    def export_to_csv(df, output_path, index=False):
        """Export to CSV"""
        df.to_csv(output_path, index=index)
        print(f"✓ Data exported to {output_path}")
    
    @staticmethod
    def export_to_excel(df_dict, output_path):
        """Export multiple dataframes to Excel sheets"""
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            for sheet_name, df in df_dict.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"✓ Data exported to {output_path}")
    
    @staticmethod
    def create_summary_report(df):
        """Create summary statistics report"""
        summary = {
            'Total Records': len(df),
            'Date Range': f"{df['date'].min()} to {df['date'].max()}",
            'Minerals': df['mineral'].nunique(),
            'Total Import Value (₹ Cr)': df['import_value'].sum(),
            'Total Export Value (₹ Cr)': df['export_value'].sum(),
            'Trade Deficit (₹ Cr)': df['import_value'].sum() - df['export_value'].sum()
        }
        
        return pd.DataFrame([summary])

============================================================================
SECTION 5: MAIN PREPROCESSING PIPELINE
============================================================================

In [26]:
def preprocess_dgcis_data(input_file, output_dir='./processed_data/'):
    """Complete preprocessing pipeline"""
    
    print("="*70)
    print("DGCI&S DATA PREPROCESSING PIPELINE")
    print("="*70)
    
    # Step 1: Load data
    loader = DGCISDataLoader()
    
    if input_file.endswith('.xlsx') or input_file.endswith('.xls'):
        df = loader.load_excel(input_file)
    else:
        df = loader.load_csv(input_file)
    
    if df is None:
        print("✗ Failed to load data")
        return None
    
    # Step 2: Inspect
    loader.inspect_data(df)
    
    # Step 3: Clean
    cleaner = DGCISDataCleaner()
    df = cleaner.standardize_columns(df)
    df = cleaner.clean_hsn_codes(df)
    df = cleaner.parse_dates(df)
    df = cleaner.clean_numeric_values(df)
    df = cleaner.handle_missing_values(df, strategy='interpolate')
    
    # Step 4: Transform
    transformer = DGCISDataTransformer()
    df_minerals = transformer.filter_critical_minerals(df)
    df_agg = transformer.aggregate_by_time(df_minerals, freq='M')
    df_pivot = transformer.pivot_import_export(df_agg)
    df_final = transformer.calculate_growth_rates(df_pivot)
    df_final = transformer.create_features(df_final)
    
    # Step 5: Country Analysis
    analyzer = CountryAnalyzer()
    
    country_insights = {}
    for mineral in ['Copper', 'Lithium', 'Graphite']:
        top_partners = analyzer.get_top_partners(df_minerals, mineral)
        concentration = analyzer.calculate_concentration_index(df_minerals, mineral)
        
        country_insights[mineral] = {
            'top_partners': top_partners,
            'concentration': concentration
        }
    
    # Step 6: Export
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    exporter = DataExporter()
    
    # Export main data
    exporter.export_to_csv(df_final, f'{output_dir}/processed_exim_data.csv')
    
    # Export to Excel with multiple sheets
    export_dict = {
        'Main_Data': df_final,
        'Summary': exporter.create_summary_report(df_final)
    }
    
    for mineral in ['Copper', 'Lithium', 'Graphite']:
        mineral_data = df_final[df_final['mineral'] == mineral]
        export_dict[f'{mineral}_Data'] = mineral_data
    
    exporter.export_to_excel(export_dict, f'{output_dir}/exim_analysis.xlsx')
    
    print("\n" + "="*70)
    print("PREPROCESSING COMPLETED SUCCESSFULLY")
    print("="*70)
    print(f"\nProcessed files saved to: {output_dir}")
    
    return df_final, country_insights

============================================================================
EXAMPLE USAGE
============================================================================

In [27]:
if __name__ == "__main__":
    # Example with sample data
    print("Creating sample DGCI&S data for demonstration...")
    
    # Create sample data
    np.random.seed(42)
    dates = pd.date_range('2017-04-01', '2024-03-01', freq='M')
    
    sample_data = []
    minerals = {
        '2603': 'Copper',
        '2530': 'Lithium',
        '2508': 'Graphite'
    }
    
    trade_types = ['Import', 'Export']
    countries = {
        'Copper': ['Chile', 'Australia', 'Indonesia', 'Peru', 'USA'],
        'Lithium': ['Australia', 'Chile', 'China', 'Argentina', 'Zimbabwe'],
        'Graphite': ['China', 'Madagascar', 'Mozambique', 'Sri Lanka', 'Canada']
    }
    
    for date in dates:
        for hsn, mineral in minerals.items():
            for trade_type in trade_types:
                for country in countries[mineral]:
                    value = np.random.uniform(100, 5000)
                    quantity = np.random.uniform(1000, 50000)
                    
                    sample_data.append({
                        'HSN': hsn,
                        'Commodity': f'{mineral} ores',
                        'Country': country,
                        'Year': str(date.year),
                        'Month': str(date.month),
                        'Trade Type': trade_type,
                        'Value (Rs Cr)': value,
                        'Quantity': quantity,
                        'Unit': 'MT'
                    })
    
    df_sample = pd.DataFrame(sample_data)
    df_sample.to_csv('sample_dgcis_data.csv', index=False)
    
    print("✓ Sample data created: sample_dgcis_data.csv")
    
    # Run preprocessing
    df_processed, insights = preprocess_dgcis_data('sample_dgcis_data.csv')
    
    print("\n✓ Pipeline execution complete!")
    print(f"✓ Processed {len(df_processed)} records")
    print("\nSample of processed data:")
    print(df_processed.head())

Creating sample DGCI&S data for demonstration...
✓ Sample data created: sample_dgcis_data.csv
DGCI&S DATA PREPROCESSING PIPELINE
Loading data from: sample_dgcis_data.csv
✓ Loaded 2490 rows, 9 columns

DATA INSPECTION REPORT

Shape: (2490, 9)

Column Names:
1. HSN
2. Commodity
3. Country
4. Year
5. Month
6. Trade Type
7. Value (Rs Cr)
8. Quantity
9. Unit

Data Types:
HSN                int64
Commodity         object
Country           object
Year               int64
Month              int64
Trade Type        object
Value (Rs Cr)    float64
Quantity         float64
Unit              object
dtype: object

Missing Values:
Series([], dtype: int64)

First 5 rows:
    HSN    Commodity    Country  Year  Month Trade Type  Value (Rs Cr)  \
0  2603  Copper ores      Chile  2017      4     Import    1935.246582   
1  2603  Copper ores  Australia  2017      4     Import    3686.770315   
2  2603  Copper ores  Indonesia  2017      4     Import     864.491338   
3  2603  Copper ores       Peru  2017  

ModuleNotFoundError: No module named 'openpyxl'